In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import os
os.environ["OMP_NUM_THREADS"] = "1" # avoid fuite de mémoire sur windows

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import  silhouette_score


In [2]:
df = pd.read_csv('c/df.csv')
print('Lines / Columns : ', df.shape)
t1 = pd.read_csv('c/t1.csv')
print('Lines / Columns : ', t1.shape)

C:\Users\LN\AppData\Local\Temp\ipykernel_87716\2106499970.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('c/df.csv')


Lines / Columns :  (18706779, 8)
Lines / Columns :  (3207, 11)


In [3]:
t1["pickup_date"] = pd.to_datetime(t1["pickup_date"])
t1['month'] = t1["pickup_date"].dt.month
t1["hour"] = t1["pickup_date"].dt.hour
t1["dayofweek"] = t1["pickup_date"].dt.dayofweek
t1['pickup_date'].dtype

dtype('<M8[ns]')

In [4]:
t1.head(1)

,pickup_date,lat,lon,dispatching_base_num,locationid,borough,zone,affiliated_base_num,month,hour,dayofweek
0,2015-05-17 10:00:00,40.756729,-73.965146,b02617,229,manhattan,sutton place/turtle bay north,b02617,5,10,6


# Remove useless variables

In [5]:
# On jette les colonnes inutiles

useless_cols = ['locationid', 'pickup_date', 'affiliated_base_num']

print("Les colonnes suivantes vont être jetées : ", useless_cols)
t1 = t1.drop(useless_cols, axis=1)
print("...Terminé.")
print(df.head())
t1.head(1)

Les colonnes suivantes vont être jetées :  ['locationid', 'pickup_date', 'affiliated_base_num']
...Terminé.
           pickup_date      lat      lon dispatching_base_num  locationid  \
0  2014-04-01 00:11:00  40.7690 -73.9549               b02512         140   
1  2014-04-01 00:21:00  40.7316 -73.9873               b02512          79   
2  2014-04-01 00:28:00  40.7588 -73.9776               b02512         161   
3  2014-04-01 00:33:00  40.7594 -73.9722               b02512         162   
4  2014-04-01 00:39:00  40.7223 -73.9887               b02512         148   

     borough             zone affiliated_base_num  
0  manhattan  lenox hill east                 NaN  
1  manhattan     east village                 NaN  
2  manhattan   midtown center                 NaN  
3  manhattan     midtown east                 NaN  
4  manhattan  lower east side                 NaN  


,lat,lon,dispatching_base_num,borough,zone,month,hour,dayofweek
0,40.756729,-73.965146,b02617,manhattan,sutton place/turtle bay north,5,10,6


# Normalize dataset

Before applying K-Means, let's normalize our dataset to avoid inflating our dataset since K-Means is using Euclidian Distance.

In [6]:
# Création du pipeline pour les variables quantitatives
numeric_features = ['lon', 'lat', 'month', 'hour', 'dayofweek'] # Positions des colonnes quantitatives dans X
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()) # pour normaliser les variables
])

# Création du pipeline pour les variables catégorielles
categorical_features = ['dispatching_base_num', 'borough',	'zone'] # Positions des colonnes catégorielles dans X
categorical_transformer = Pipeline(
    steps=[
    ('encoder', OneHotEncoder()) # on encode les catégories sous forme de colonnes comportant des 0 et des 1
    ])

# On combine les pipelines dans un ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Preprocessings sur le dataset
print("Preprocessing sur le train set...")
print(t1.head())
X = preprocessor.fit_transform(t1) # fit_transform !!
print('...Terminé.')
print(X[0:5, :])
print()

Preprocessing sur le train set...
         lat        lon dispatching_base_num    borough  \
0  40.756729 -73.965146               b02617  manhattan   
1  40.766948 -73.959635               b02617  manhattan   
2  40.745379 -73.948892               b02617     queens   
3  40.718804 -73.957418               b02617   brooklyn   
4  40.766948 -73.959635               b02617  manhattan   

                             zone  month  hour  dayofweek  
0   sutton place/turtle bay north      5    10          6  
1                 lenox hill west      5    10          6  
2  long island city/hunters point      5    10          6  
3       williamsburg (north side)      5    10          6  
4                 lenox hill west      5    10          6  
...Terminé.
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 25 stored elements and shape (5, 220)>
  Coords	Values
  (0, 0)	-0.1378871412238405
  (0, 1)	0.2909177057177657
  (0, 7)	1.0
  (0, 13)	1.0
  (0, 188)	1.0
  (1, 0)	-0.01809661481

In [7]:
# from sklearn.cluster import KMeans

# # Instanciate KMeans with k=4 and initialisation with k-means++
# # You should always use k-means++ as it alleviate the problem of local minimum convergence
# kmeans = KMeans(n_clusters=4, random_state=42)

# # Fit kmeans to our dataset
# kmeans.fit(X)

In [8]:
# data = t1.assign(cluster=kmeans.labels_.astype(str)) # transform les cluster en labels
# centers = data.groupby("cluster")[["lat", "lon"]].median().reset_index()
# fig = px.scatter_map(data, lat="lat", lon="lon", color="cluster", zoom=9, map_style="carto-positron", category_orders={"cluster": ["0", "1", "2", "3"]})
# fig.add_scattermap(lat=centers["lat"], lon=centers["lon"], mode="markers", marker=dict(size=12, color="black"), name="centres")
# fig.show()

# determine k

In [9]:
wcss =  []
for i in range (2,11):
    kmeans = KMeans(n_clusters= i, random_state=42)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

print(wcss)

[10855.036960859616, 8597.048166506516, 7696.871021028934, 7473.753796475606, 6875.74290428299, 6684.378046898118, 5994.249196799646, 5764.320173641099, 5587.50504220478]


In [10]:
fig = px.line(x = range(2,11), y = wcss)
fig.show()

In [11]:
# Utilisation du silhouette_score pour déterminer le nombre optimal de clusters
s_score = []
for i in range (2,11):
    kmeans = KMeans(n_clusters= i)
    kmeans.fit(X)
    s_score.append(silhouette_score(X, kmeans.predict(X)))

print(s_score)


[0.23637160942675212, 0.2643112447853653, 0.19777313126046603, 0.18061117607117733, 0.14555399151302179, 0.15320886882125556, 0.1975714762457263, 0.17780405640004654, 0.17171689600699266]


In [12]:
# Utilisation du silhouette_score pour déterminer le nombre optimal de clusters
s_score = []
for i in range (2,11):
    kmeans = KMeans(n_clusters= i)
    kmeans.fit(X)
    s_score.append(silhouette_score(X, kmeans.predict(X)))

print(s_score)

[0.30370900924410843, 0.2643112447853653, 0.20647799939848077, 0.17095430159647398, 0.15575812043158466, 0.18438748481232667, 0.18802981054080528, 0.17643719568863994, 0.17247544817594918]


In [13]:
# Affichage de scores en fonction du nombre de clusters
fig = px.bar(x = range(2,11), y = s_score)
fig.show()

# Fit kmeans with 4 clusters

In [15]:
# On ré-entraîne un KMeans avec le nombre optimal de clusters
kmeans = KMeans(n_clusters=4, random_state=42)

# Fit kmeans to our dataset
kmeans.fit(X)

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",4
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary `.",42
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'


In [16]:
t1.loc[:,'Cluster_KMeans'] = kmeans.predict(X)
t1.head()

,lat,lon,dispatching_base_num,borough,zone,month,hour,dayofweek,Cluster_KMeans
0,40.756729,-73.965146,b02617,manhattan,sutton place/turtle bay north,5,10,6,1
1,40.766948,-73.959635,b02617,manhattan,lenox hill west,5,10,6,1
2,40.745379,-73.948892,b02617,queens,long island city/hunters point,5,10,6,1
3,40.718804,-73.957418,b02617,brooklyn,williamsburg (north side),5,10,6,3
4,40.766948,-73.959635,b02617,manhattan,lenox hill west,5,10,6,1


In [17]:
data = t1.assign(cluster=kmeans.labels_.astype(str)) # transform les cluster en labels
centers = data.groupby("cluster")[["lat", "lon"]].median().reset_index()
fig = px.scatter_map(data, lat="lat", lon="lon", color="cluster", zoom=9, map_style="carto-positron", category_orders={"cluster": ["0", "1", "2", "3"]})
fig.add_scattermap(lat=centers["lat"], lon=centers["lon"], mode="markers", marker=dict(size=12, color="black"), name="centres")
fig.show()

In [18]:
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(X)

t1.loc[:, "kmeans_cluster"] = kmeans.predict(X)
t1.head()

,lat,lon,dispatching_base_num,borough,zone,month,hour,dayofweek,Cluster_KMeans,kmeans_cluster
0,40.756729,-73.965146,b02617,manhattan,sutton place/turtle bay north,5,10,6,1,1
1,40.766948,-73.959635,b02617,manhattan,lenox hill west,5,10,6,1,1
2,40.745379,-73.948892,b02617,queens,long island city/hunters point,5,10,6,1,1
3,40.718804,-73.957418,b02617,brooklyn,williamsburg (north side),5,10,6,3,3
4,40.766948,-73.959635,b02617,manhattan,lenox hill west,5,10,6,1,1


In [19]:
t1["kmeans_cluster"].value_counts()

kmeans_cluster
1    1885
3     734
0     346
2     242
Name: count, dtype: int64

In [20]:
map_data = t1.copy()
map_data = map_data.dropna(subset=["lat", "lon", "Cluster_KMeans"])

In [21]:
cluster_colors = {
    0: "#FF6B6B",  # rouge
    1: "#19B0DE",  # bleu
    2: "#17CE48",  # vert
    3: "#F5A623",  # orange
}

In [22]:
fig = go.Figure()
for cluster_id, color in cluster_colors.items():
    df_cluster = map_data[map_data["Cluster_KMeans"] == cluster_id]

    fig.add_trace(
        go.Scattermap(
            lat=df_cluster["lat"],
            lon=df_cluster["lon"],
            mode="markers",
            marker={"size": 8, "color": color},
            customdata=df_cluster[["borough", "zone", "dayofweek", "lat", "lon"]].values,
            hovertemplate=(
                "Borough: %{customdata[0]}<br>"
                "Zone: %{customdata[1]}<br>"
                "Jour: %{customdata[2]}<br>"
                "Lat: %{customdata[3]:.4f}<br>"
                "Lon: %{customdata[4]:.4f}<extra></extra>"
            ),
            name=f"Cluster {cluster_id}",
        )
    )
fig.update_layout(
map_style="carto-positron",
map_center={"lat": 40.73, "lon": -73.97},
map_zoom=10,
height=600,
title="Prises en charge sur 10 jours en fonction des clusters KMeans",
)

fig.show()

# Db SCAN

In [23]:
from sklearn.cluster import DBSCAN
import numpy as np

In [24]:
Xb = t1[numeric_features]
X_scaled = StandardScaler().fit_transform(Xb)
dbscan_b = DBSCAN(eps=0.3, min_samples=100, metric="euclidean") # Le 4e cluster est considéré comme du bruit
dbscan_b.fit(X_scaled)
t1["dbscan_cluster"] = dbscan_b.labels_
t1["dbscan_cluster"].value_counts()

dbscan_cluster
 0    1844
-1     970
 1     179
 2     111
 3     103
Name: count, dtype: int64

In [25]:
t1.head()

,lat,lon,dispatching_base_num,borough,zone,month,hour,dayofweek,Cluster_KMeans,kmeans_cluster,dbscan_cluster
0,40.756729,-73.965146,b02617,manhattan,sutton place/turtle bay north,5,10,6,1,1,0
1,40.766948,-73.959635,b02617,manhattan,lenox hill west,5,10,6,1,1,0
2,40.745379,-73.948892,b02617,queens,long island city/hunters point,5,10,6,1,1,-1
3,40.718804,-73.957418,b02617,brooklyn,williamsburg (north side),5,10,6,3,3,1
4,40.766948,-73.959635,b02617,manhattan,lenox hill west,5,10,6,1,1,0


In [26]:
map_data = t1.copy()
map_data = map_data.dropna(subset=["dbscan_cluster"])

In [27]:
cluster_colors = {
    0: "#FF6B6B",  # rouge
    1: "#19B0DE",  # bleu
    2: "#17CE48",  # vert
    3: "#F5A623",  # orange
}

In [28]:
print("Pickups in the selected hour:", len(t1))

fig = go.Figure()

for cluster_id, color in cluster_colors.items():
    df_cluster = map_data[map_data["dbscan_cluster"] == cluster_id]

    fig.add_trace(
        go.Scattermap(
            lat=df_cluster["lat"],
            lon=df_cluster["lon"],
            mode="markers",
            marker={"size": 8, "color": color},
            customdata=df_cluster[["borough", "zone", "hour", "dayofweek", "lat", "lon"]
            ].values,
            hovertemplate=(
                "Borough: %{customdata[0]}<br>"
                "Zone: %{customdata[1]}<br>"
                "Date: %{customdata[2]}<br>"
                "Jour: %{customdata[3]}<br>"
                "Lat: %{customdata[4]:.4f}<br>"
                "Lon: %{customdata[5]:.4f}<extra></extra>"
            ),
            name=f"Cluster {cluster_id}",
        )
    )

fig.update_layout(
    map_style="carto-positron",
    map_center={"lat": 40.73, "lon": -73.97},
    map_zoom=10,
    height=600,
    title="Prises en charge sur 10 jours en fonction des clusters DBscan",
)

fig.show()

Pickups in the selected hour: 3207


In [29]:
fig = go.Figure()
for cluster_id, color in cluster_colors.items():
    df_cluster = map_data[map_data["dbscan_cluster"] == cluster_id]
    fig.add_trace(
        go.Scattermap(
            lat=df_cluster["lat"],
            lon=df_cluster["lon"],
            mode="markers",
            marker=dict(color=color),
            name=f"Cluster {cluster_id}",
        )
    )
fig.show()

Sur le modèle de DBsan, la méthode clustering ne parvient pas à indentifier des clusters pertinents et conserve 1 cluster qui est considéré comme du bruit. Il est donc préférable de se baser sur les clusters Kmeans pour l'analyse.

In [30]:
print("Pickups in the selected hour:", len(t1))

fig = go.Figure()

for cluster_id, color in cluster_colors.items():
    df_cluster = map_data[map_data["kmeans_cluster"] == cluster_id]

    fig.add_trace(
        go.Scattermap(
            lat=df_cluster["lat_x"],
            lon=df_cluster["lon_x"],
            mode="markers",
            marker={"size": 8, "color": color},
            customdata=df_cluster[
                ["borough", "zone", "date/time", "weekday", "lat_x", "lon_x"]
            ].values,
            hovertemplate=(
                "Borough: %{customdata[0]}<br>"
                "Zone: %{customdata[1]}<br>"
                "Date: %{customdata[2]}<br>"
                "Jour: %{customdata[3]}<br>"
                "Lat: %{customdata[4]:.4f}<br>"
                "Lon: %{customdata[5]:.4f}<extra></extra>"
            ),
            name=f"Cluster {cluster_id}",
        )
    )

fig.update_layout(
    map_style="carto-positron",
    map_center={"lat": 40.73, "lon": -73.97},
    map_zoom=10,
    height=600,
    title="Prises en charge sur 10 jours en fonction des clusters KMeans",
)

fig.show()

Pickups in the selected hour: 3207


KeyError: 'lat_x'

Le modèle de Kmeans parvient a identifier les 4 clusters principaux concernant les prises en charge de clients. Le modèle semble attribuer une plus forte importance aux indications géographiques qu'aux jours de la semaine pour déterminer les clusters.